<div style="background:linear-gradient(135deg,#15151C 0%,#0D0D12 100%); border-radius:16px; padding:22px 26px; border:1px solid #8B5CF644; box-shadow:0 0 30px #8B5CF633; font-family:'Poppins','Segoe UI',system-ui,sans-serif;">
<h1 style="margin:0; font-size:28px; font-weight:700; background:linear-gradient(90deg,#A855F7,#EC4899); -webkit-background-clip:text; background-clip:text; color:transparent;">Who remembers the meeting?</h1>
<p style="margin:6px 0 0 0; color:#9CA3AF; font-size:14px;">Agenda item 1, 00:00-00:08. Record the skit, press one button, show the minutes.</p>
</div>

For the person running the hook. Participants don't use this file. Their notebook is `audio_workshop.ipynb`.

Run section 1 before the session. It loads the Whisper model, which takes a minute or two. Don't do it live.

During the hook:
1. Section 2: record the skit from the browser mic (a live waveform and frequency display while you talk).
2. Section 3: press the button on that recording.
3. Section 4: use the backup WAV if recording fails.
4. Section 5: text-only fallback if Whisper won't load.

The live skit is not rehearsed or tested end to end. Only the backup WAV is verified. Rehearse the live path first (checklist in `demo/FACILITATOR.md`). Until then, use the backup WAV.

Pacing for a live skit:
- Say each turn in one go. No long pause inside it.
- Two turns have two sentences. Say both without stopping.
- Leave a clear 2-second pause before the other person speaks.
- Don't overlap.

The transcriber starts a new speaker at a 1.5-second gap. A quick back-and-forth merges into one speaker. A long pause inside a turn looks like a speaker change. The recorder below shows a small "pause > 1.5s = next speaker" meter while you talk, so you can feel the gap instead of guessing it.

What to say and when: `demo/FACILITATOR.md`.

If this notebook or Colab itself won't cooperate, `python demo/hook_demo_cli.py` runs the same demo from a terminal, no Colab needed (see its own docstring).

## 1. Setup: run before the session
On Colab: Runtime > Change runtime type > T4 GPU. It makes Whisper faster.
Upload `facilitator_files.zip` to the Files panel (left) first.

In [ ]:
import os, sys, shutil, subprocess, importlib.util, zipfile
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
import warnings; warnings.filterwarnings("ignore", message="FP16")  # Whisper on CPU only

if importlib.util.find_spec("whisper") is None:
    print("Installing Whisper, about a minute...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai-whisper"], check=True)
if os.path.exists("facilitator_files.zip") and not os.path.exists("prototype"):
    zipfile.ZipFile("facilitator_files.zip").extractall(".")
PROTO = next((p for p in ("prototype", "../prototype") if os.path.isdir(p)), None)
PROTO or sys.exit("Upload facilitator_files.zip to the Files panel (left), then run this cell again.")
sys.path.insert(0, PROTO)
DEMO_DIR = next((p for p in ("demo", "../demo") if os.path.isdir(p)), None)
DEMO_DIR or sys.exit("Upload facilitator_files.zip to the Files panel (left), then run this cell again.")
sys.path.insert(0, DEMO_DIR)
print("ffmpeg:", "found" if shutil.which("ffmpeg") else "NOT FOUND")

# medium.en is fast on a T4. On CPU it takes about 3 minutes (measured), so use small.en:
# same minutes, about 1 minute.
import torch
has_gpu = torch.cuda.is_available()
print(f"GPU: {'yes' if has_gpu else 'no'} -- expect the button press to take "
      f"{'under a minute' if has_gpu else 'about 1 minute (small.en, no GPU here)'}")
os.environ["WHISPER_MODEL"] = "medium.en" if has_gpu else "small.en"

# Preload now, not during the hook. The first Whisper call downloads and loads the model.
import whisper
import transcriber
transcriber._model = whisper.load_model(transcriber.WHISPER_MODEL)
print(f"Whisper ({transcriber.WHISPER_MODEL}) preloaded and ready.")

# Cells 6, 8 and 10 need these. Imported once here, so each cell works on its own.
from demo import run_demo, run_demo_staged
from IPython.display import Audio, HTML, Markdown, display

# hook_ui: the live recorder (Record/Stop, canvas, silence meter).
# hook_theme: the banner, progress strip, minutes cards, and the dark
# waveform/spectrogram playback panel. Presentation only -- neither touches
# labelling or minutes text, both just render what run_demo_staged() returns.
import hook_ui
import hook_theme
display(hook_theme.banner())

## 2. Record the skit from the browser microphone
Click **Record**. Do the skit (two presenters, `demo/script.txt`). Say each turn in one go. Pause 2 seconds before the other person speaks -- watch the silence meter fill as you pause (see the pacing note above). Click **Stop** when done. The recording saves to `skit.wav`.

Needs Colab. On a laptop, record with any app and pass that file's path to `python demo/hook_demo_cli.py your_recording.wav` instead (see its docstring) -- that's a separate, non-notebook path if this one isn't cooperating.

In [ ]:
skit_path = hook_ui.record_with_ui(out_path="skit.wav", title="Record the skit")
if skit_path:
    hook_ui.dark_recording_panel(skit_path, title="Your recording")

## 3. Press. One. Button. (the live recording)
Watch the progress strip -- Listening, Transcribing, Labelling, Writing minutes -- then the minutes cards.

In [ ]:
if skit_path:
    update, finish = hook_ui.start_progress_strip()
    minutes = run_demo_staged(skit_path, banner=False, on_stage=update)
    finish()
    hook_ui.render_minutes_cards(minutes)
else:
    print("No recording from cell 4 -- use the backup WAV instead (cells 8, 10).")

## 4. Backup: the two-voice WAV
Use this if the recording failed. See `demo/make_backup_wav.py` for how it was made and `demo/FACILITATOR.md` for when to use it.

Two cells. Run them in order, with a pause between:
1. The next cell shows the waveform and plays the WAV to the room.
2. The cell under 4b presses the button.

The room calls out the decisions and to-dos before the minutes appear.

In [ ]:
backup_path = next(p for p in ("demo/backup_meeting.wav", "../demo/backup_meeting.wav") if os.path.isfile(p))
hook_ui.dark_recording_panel(backup_path, title="Backup recording (play to the room)")
# Let them call out the decisions and to-dos before the next cell presses the button.

## 4b. Press the button on the backup
Run this only after the room has called out the decisions and to-dos.

In [ ]:
update, finish = hook_ui.start_progress_strip()
minutes = run_demo_staged(backup_path, banner=False, on_stage=update)
finish()
hook_ui.render_minutes_cards(minutes)

## 5. Text-only fallback
Use this if Whisper won't load. It runs `demo/script.txt` as typed text through the same minutes code. No audio, no model, instant.

It isn't the real skit, but it keeps the hook going.

In [ ]:
update, finish = hook_ui.start_progress_strip()
minutes = run_demo_staged(banner=False, on_stage=update)   # no argument: demo/script.txt, no Whisper at all
finish()
hook_ui.render_minutes_cards(minutes)